# Day 11: GQA、MQA、MHA

Query heads 数不变，KV heads 分别为 1、2、H。缓存紧凑 KV，不保存扩展副本。改变 KV heads 会改变参数形状，不能要求不同随机模型输出相同。

参考源码：`learning/mini_transformer/model.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY08_14.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 11完整讲义](../docs/DAY11_GQA_MQA.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

固定Hq=4，把独立KV头数从4改成2或1：

```text
MHA： Q0→KV0，Q1→KV1，Q2→KV2，Q3→KV3
GQA： Q0/Q1→KV0，Q2/Q3→KV1
MQA： Q0/Q1/Q2/Q3→KV0
```

共享资料柜不意味着四个问题答案相同：各Q不同，权重也可不同。
固定L=2、B=1、T=8、D=8、FP32：
紧凑总KV为 `2*L*B*T*Hkv*D*4` bytes，Hkv=4/2/1分别为4096/2048/1024 bytes。
这只是KV数组，不是总参数或完整进程峰值。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(11, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

KV 字节数为何与 KV heads 成正比？共享的是权重/表示还是 Q？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：四个Q共享一个KV，Query参数也变少吗？**
答：Hq和D不变时Q投影宽度不变，改变的是K/V生成与缓存头数。

**问：Hkv减半是否模型总内存减半？**
答：不是，仅相同前提下的紧凑KV部分减半，其他权重和buffer另计。

**问：为什么不拿三种模型的logits直接相等作为验收？**
答：结构和权重不同；应各自做full/cached一致性，再另行评估质量。

**问：GQA等同MLA吗？**
答：不是，MLA改变联合latent表示与计算路径，不只是减少KV头数。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY11_GQA_MQA.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。